In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


In [2]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

CUDA available: True
CUDA device: NVIDIA A100 80GB PCIe


In [3]:
# First, let's explore the original repo and find documentation files
import os

original_repo = '/net/scratch2/smallyan/function_vectors_eval'
replication_dir = '/net/scratch2/smallyan/function_vectors_eval/evaluation/replications'

# List files in original repo
print("Files in original repo:")
for item in os.listdir(original_repo):
    print(f"  {item}")
    
print("\n" + "="*50)
print("\nFiles in evaluation/replications:")
if os.path.exists(replication_dir):
    for item in os.listdir(replication_dir):
        print(f"  {item}")
else:
    print("  Directory does not exist")

Files in original repo:
  no_exe_evaluation
  .gitignore
  doc_only_evaluation
  src
  fv_overview.png
  .git
  documentation.pdf
  notebooks
  dataset_files
  plan.md
  CodeWalkthrough.md
  evaluation
  fv_environment.yml


Files in evaluation/replications:
  self_replication_evaluation.json
  evaluation_replication.md
  replication.ipynb
  documentation_replication.md


In [4]:
# Let's look for documentation.md in original repo
# The task mentions documentation.md or equivalent

# Check if there's a documentation.md or README
doc_files = ['documentation.md', 'README.md', 'documentation.pdf', 'CodeWalkthrough.md', 'plan.md']
for doc in doc_files:
    path = os.path.join(original_repo, doc)
    if os.path.exists(path):
        print(f"Found: {path}")
        
# Let's also check if there's a documentation folder
for item in os.listdir(original_repo):
    item_path = os.path.join(original_repo, item)
    if os.path.isdir(item_path):
        print(f"\nDirectory: {item}")
        sub_items = os.listdir(item_path)[:10]  # First 10 items
        for sub in sub_items:
            print(f"  {sub}")

Found: /net/scratch2/smallyan/function_vectors_eval/documentation.pdf
Found: /net/scratch2/smallyan/function_vectors_eval/CodeWalkthrough.md
Found: /net/scratch2/smallyan/function_vectors_eval/plan.md

Directory: no_exe_evaluation
  replications
  code_critic_evaluation.ipynb
  generalization_eval.ipynb
  generalization_eval_summary.json
  code_critic_summary.json

Directory: doc_only_evaluation
  replication_evaluation.md
  code_critic_summary.json
  self_replication_evaluation.json
  self_matching.ipynb
  generalization_eval_summary.json
  generalization_eval.ipynb
  consistency_evaluation.json
  code_critic_evaluation.ipynb

Directory: src
  __pycache__
  portability_eval.py
  utils
  test_numheads.py
  compute_indirect_effect.py
  vocab_reconstruction.py
  __init__.py
  eval_scripts
  compute_avg_hidden_state.py
  natural_text_eval.py

Directory: .git
  refs
  info
  COMMIT_EDITMSG
  config
  hooks
  description
  logs
  objects
  packed-refs
  FETCH_HEAD

Directory: notebooks
  fv

In [5]:
# Let's read the CodeWalkthrough.md as the original documentation
with open(os.path.join(original_repo, 'CodeWalkthrough.md'), 'r') as f:
    original_doc = f.read()
print("Original CodeWalkthrough.md:")
print("="*80)
print(original_doc)

Original CodeWalkthrough.md:
# Function Vectors in Large Language Models
### [Project Website](https://functions.baulab.info) | [Arxiv Preprint](https://arxiv.org/abs/2310.15213) | [OpenReview](https://openreview.net/forum?id=AwyxtyMwaG)

This repository contains data and code for the paper: [Function Vectors in Large Language Models](https://arxiv.org/pdf/2310.15213).

<p align="left">
<img src="https://functions.baulab.info/images/Paper/fv-demonstrations.png" style="width:100%;"/>
</p> 

## Setup

We recommend using conda as a package manager. 
The environment used for this project can be found in the `fv_environment.yml` file.
To install, you can run: 
```
conda env create -f fv_environment.yml
conda activate fv
```

## Demo Notebook
Checkout `notebooks/fv_demo.ipynb` for a jupyter notebook with a demo of how to create a function vector and use it in different contexts.

## Data
The datasets used in our project can be found in the `dataset_files` folder.

## Code
Our main evaluation

In [6]:
# Let's also look at the plan.md and fv_demo notebook to understand the original experiment results
with open(os.path.join(original_repo, 'plan.md'), 'r') as f:
    plan_doc = f.read()
print("Original plan.md:")
print("="*80)
print(plan_doc)

Original plan.md:
# Plan
## Objective
To investigate whether autoregressive transformer language models develop compact vector representations of input-output functions (called function vectors) within their hidden states during in-context learning, and to characterize these representations across diverse tasks and models.

## Hypothesis
1. A small number of attention heads transport a compact representation of the demonstrated task (function vector) that is robust to changes in context and can trigger task execution in zero-shot and natural text settings.
2. Function vectors contain information encoding the output space of the function, but this information alone is not sufficient to reconstruct a working function vector.
3. Function vectors can be composed through vector algebra to create vectors that trigger new complex tasks combining constituent tasks.

## Methodology
1. Apply causal mediation analysis to identify attention heads with highest average indirect effect (AIE) across d

In [7]:
# Now let's read the demo notebook to understand the original expected results
import json

demo_notebook_path = os.path.join(original_repo, 'notebooks/fv_demo.ipynb')
with open(demo_notebook_path, 'r') as f:
    demo_notebook = json.load(f)

# Print all cells with their outputs
print("Demo Notebook Contents:")
print("="*80)
for i, cell in enumerate(demo_notebook['cells']):
    cell_type = cell['cell_type']
    source = ''.join(cell['source'])
    print(f"\n--- Cell {i} ({cell_type}) ---")
    print(source[:2000])  # Truncate long cells
    
    if 'outputs' in cell and cell['outputs']:
        print("\n[Output]:")
        for output in cell['outputs'][:3]:  # First 3 outputs
            if 'text' in output:
                output_text = ''.join(output['text'])
                print(output_text[:1000])

Demo Notebook Contents:

--- Cell 0 (code) ---
%load_ext autoreload
%autoreload 2

--- Cell 1 (code) ---
import os, re, json
import torch, numpy as np

import sys
sys.path.append('..')
torch.set_grad_enabled(False)

from src.utils.extract_utils import get_mean_head_activations, compute_universal_function_vector
from src.utils.intervention_utils import fv_intervention_natural_text, function_vector_intervention
from src.utils.model_utils import load_gpt_model_and_tokenizer
from src.utils.prompt_utils import load_dataset, word_pairs_to_prompt_data, create_prompt
from src.utils.eval_utils import decode_to_vocab, sentence_eval

--- Cell 2 (markdown) ---
## Load model & tokenizer

--- Cell 3 (code) ---
model_name = 'EleutherAI/gpt-j-6b'
model, tokenizer, model_config = load_gpt_model_and_tokenizer(model_name)
EDIT_LAYER = 9

--- Cell 4 (markdown) ---
## Load dataset and Compute task-conditioned mean activations

--- Cell 5 (code) ---
dataset = load_dataset('antonym', seed=0)
mean_activations

In [8]:
# Now let's read the replicated documentation
replication_doc_path = os.path.join(replication_dir, 'documentation_replication.md')
with open(replication_doc_path, 'r') as f:
    replicated_doc = f.read()
    
print("Replicated documentation_replication.md:")
print("="*80)
print(replicated_doc)

Replicated documentation_replication.md:
# Function Vectors in Large Language Models - Replication Documentation

## Goal

This replication aims to verify the core claims of the "Function Vectors in Large Language Models" paper (Todd et al., ICLR 2024), which investigates whether autoregressive transformer language models develop compact vector representations of input-output functions within their hidden states during in-context learning.

The specific goals are:
1. Extract function vectors from ICL prompts by averaging attention head activations
2. Test function vector portability across different contexts (ICL, shuffled-label, zero-shot, natural text)
3. Verify that function vectors can trigger task execution without explicit in-context examples

## Data

The replication uses the datasets provided in the repository's `dataset_files/` directory:

- **Antonym task** (`abstractive/antonym.json`): 2398 word pairs (e.g., "hot" → "cold")
  - Train: 1678 pairs
  - Valid: 216 pairs
  - Test

In [9]:
# Let's also check the replication notebook to see actual execution results
replication_notebook_path = os.path.join(replication_dir, 'replication.ipynb')
with open(replication_notebook_path, 'r') as f:
    replication_notebook = json.load(f)

# Print cells and their outputs
print("Replication Notebook Contents:")
print("="*80)

for i, cell in enumerate(replication_notebook['cells']):
    cell_type = cell['cell_type']
    source = ''.join(cell['source'])
    
    # Only print cells with outputs or meaningful code
    if cell_type == 'code' and ('outputs' in cell and cell['outputs']):
        print(f"\n--- Cell {i} ({cell_type}) ---")
        print(source[:1500])
        
        print("\n[Output]:")
        for output in cell['outputs'][:5]:
            if 'text' in output:
                output_text = ''.join(output['text'])
                print(output_text[:1500])

Replication Notebook Contents:

--- Cell 0 (code) ---
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

[Output]:
Working directory: /home/smallyan/eval_agent


--- Cell 2 (code) ---
# Setup and imports
import os
import sys
import json
import torch
import numpy as np
import pandas as pd
import random
from typing import Dict, List, Tuple, Any, Optional
from collections import Counter

# Check GPU availability
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Set seed for reproducibility
def set_random_seed(seed: int = 42):
    """Set all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = 

In [10]:
# Let's get the complete output from later cells in the replication notebook
# Looking for the final summary/results

for i, cell in enumerate(replication_notebook['cells']):
    if i >= 31:  # Start from cell 31 onwards
        cell_type = cell['cell_type']
        source = ''.join(cell['source'])
        
        if 'outputs' in cell and cell['outputs']:
            print(f"\n--- Cell {i} ({cell_type}) ---")
            print(source[:1000])
            
            print("\n[Output]:")
            for output in cell['outputs']:
                if 'text' in output:
                    output_text = ''.join(output['text'])
                    print(output_text[:2000])


--- Cell 31 (code) ---
# Quantitative evaluation on country-capital (zero-shot)
set_random_seed(42)
print("Country-Capital Zero-shot + FV evaluation:")
cc_zeroshot_results = evaluate_fv_accuracy(
    cc_dataset, model, model_config, tokenizer,
    CC_FV, EDIT_LAYER,
    n_shots=0, shuffle_labels=False, max_samples=42  # All test samples
)
print(f"  Zero-shot baseline Top-1: {cc_zeroshot_results['clean_top1']:.1f}%")
print(f"  + FV intervention Top-1: {cc_zeroshot_results['interv_top1']:.1f}%")

[Output]:
Country-Capital Zero-shot + FV evaluation:

Evaluating: 100%|██████████| 42/42 [00:01<00:00, 24.94it/s]
  Zero-shot baseline Top-1: 7.1%
  + FV intervention Top-1: 83.3%





In [11]:
# Let's look at the original paper results in plan.md more closely and compare
# We need to extract the exact numbers from both documents

# From the original plan.md:
original_results = """
From plan.md (Original):
- Portability of function vectors across contexts:
  - Shuffled-label context GPT-J+FV: 90.8% vs 39.1% baseline
  - Zero-shot: 57.5% vs 5.5% baseline
  
- Performance across diverse tasks (GPT-J on 34 additional tasks):
  - Shuffled-label: 80.4%
  - Zero-shot: 46.1%
  
- Natural text portability:
  - Antonym FV: 55-68% accuracy vs 0-3% baseline
  - Country-Capital FV: 57-72% vs 4-23% baseline
"""

# From the replicated documentation:
replicated_results = """
From documentation_replication.md:

Antonym Task (50 test samples):
| Context               | Baseline | + Function Vector |
| Clean ICL (10-shot)   | 64.0%    | 62.0%             |
| Shuffled-label        | 30.0%    | 54.0%             |
| Zero-shot             | 0.0%     | 26.0%             |

Country-Capital Task (42 test samples):
| Context    | Baseline | + Function Vector |
| Zero-shot  | 7.1%     | 83.3%             |

Qualitative Results:
- Antonym natural text: Without FV: "unchanging", With FV: "dynamic"
- Country-Capital natural text: Both produce correct answer "Yaoundé"
"""

print(original_results)
print("\n" + "="*80 + "\n")
print(replicated_results)


From plan.md (Original):
- Portability of function vectors across contexts:
  - Shuffled-label context GPT-J+FV: 90.8% vs 39.1% baseline
  - Zero-shot: 57.5% vs 5.5% baseline
  
- Performance across diverse tasks (GPT-J on 34 additional tasks):
  - Shuffled-label: 80.4%
  - Zero-shot: 46.1%
  
- Natural text portability:
  - Antonym FV: 55-68% accuracy vs 0-3% baseline
  - Country-Capital FV: 57-72% vs 4-23% baseline




From documentation_replication.md:

Antonym Task (50 test samples):
| Context               | Baseline | + Function Vector |
| Clean ICL (10-shot)   | 64.0%    | 62.0%             |
| Shuffled-label        | 30.0%    | 54.0%             |
| Zero-shot             | 0.0%     | 26.0%             |

Country-Capital Task (42 test samples):
| Context    | Baseline | + Function Vector |
| Zero-shot  | 7.1%     | 83.3%             |

Qualitative Results:
- Antonym natural text: Without FV: "unchanging", With FV: "dynamic"
- Country-Capital natural text: Both produce correct a

In [12]:
# Let's also check if there's a demo output to compare against
# The prompt says "If it is replicating the demo, compare with the demo"

# Let's look at the original demo notebook outputs more carefully
print("Checking the original demo notebook outputs...")
print("="*80)

for i, cell in enumerate(demo_notebook['cells']):
    if 'outputs' in cell and cell['outputs']:
        source = ''.join(cell['source'])
        print(f"\n--- Demo Cell {i} ---")
        print(source[:500])
        print("\n[Output]:")
        for output in cell['outputs'][:3]:
            if 'text' in output:
                output_text = ''.join(output['text'])
                print(output_text[:1000])

Checking the original demo notebook outputs...


In [13]:
# The original demo notebook doesn't have outputs saved. Let's check self_replication_evaluation.json
# This might have the comparison data

self_eval_path = os.path.join(replication_dir, 'self_replication_evaluation.json')
with open(self_eval_path, 'r') as f:
    self_eval = json.load(f)
    
print("Self-replication evaluation:")
print(json.dumps(self_eval, indent=2))

Self-replication evaluation:
{
  "Checklist": {
    "RP1_ImplementationReconstructability": "PASS",
    "RP2_EnvironmentReproducibility": "PASS",
    "RP3_DeterminismAndStability": "PASS",
    "RP4_DemoPresentation": "PASS"
  },
  "Rationale": {
    "RP1_ImplementationReconstructability": "The experiment can be reconstructed from plan.md and CodeWalkthrough.md without missing steps. The plan clearly describes methodology and experiments, the code walkthrough explains utility file purposes, and the demo notebook provides a complete working example. No major guesswork was required.",
    "RP2_EnvironmentReproducibility": "The environment can be restored using fv_environment.yml. All packages (transformers, torch, baukit, sklearn, pandas, numpy) are standard and available. GPT-J 6B loads from HuggingFace, datasets are included in the repository, and no external API keys are required. No version conflicts occurred.",
    "RP3_DeterminismAndStability": "Results are stable with controlled ra

In [14]:
# Let's also check evaluation_replication.md which might have comparison details
eval_replication_path = os.path.join(replication_dir, 'evaluation_replication.md')
with open(eval_replication_path, 'r') as f:
    eval_replication = f.read()
    
print("Evaluation Replication:")
print("="*80)
print(eval_replication)

Evaluation Replication:
# Function Vectors Replication - Evaluation

## Replication Summary

This document evaluates the replication of the "Function Vectors in Large Language Models" paper (Todd et al., ICLR 2024). The replication was performed by reimplementing the core functionality from scratch based on the provided plan.md and CodeWalkthrough.md files.

## What Was Replicated

1. **Data Loading**: Implemented ICLDataset class and train/valid/test splitting
2. **Prompt Construction**: Created functions to build ICL prompts with configurable templates
3. **Model Loading**: Loaded GPT-J 6B with appropriate configuration for hooking
4. **Activation Extraction**: Implemented activation extraction at attention head outputs using baukit
5. **Function Vector Computation**: Implemented FV computation using pre-defined universal heads
6. **Intervention Mechanism**: Implemented hidden state modification during inference
7. **Evaluation**: Tested on antonym and country-capital tasks in multip

In [15]:
# Now let's conduct the evaluation based on what we've found

# EVALUATION ANALYSIS
analysis = """
## Documentation Evaluation Analysis

### DE1. Result Fidelity

**Comparison of Results:**

**Antonym Task:**
| Metric | Original (plan.md) | Replicated | Deviation |
|--------|-------------------|------------|-----------|
| Shuffled-label baseline | 39.1% | 30.0% | -23.3% relative |
| Shuffled-label + FV | 90.8% | 54.0% | -40.5% relative |
| Zero-shot baseline | 5.5% | 0.0% | -100% relative |
| Zero-shot + FV | 57.5% | 26.0% | -54.8% relative |

**Country-Capital Task:**
| Metric | Original | Replicated | Note |
|--------|----------|------------|------|
| Zero-shot baseline | Not specified | 7.1% | - |
| Zero-shot + FV | 57-72% (natural text) | 83.3% | Exceeds original |

**Assessment for DE1:**
The replicated results show the same DIRECTIONAL trends as the original:
- Function vectors improve performance over baselines in all tested contexts
- The qualitative demonstrations (antonym: "static" → "dynamic", country-capital: correct capital) match expected behavior

HOWEVER, the absolute numeric values deviate significantly (>5%) from the original:
- Antonym shuffled-label +FV: 54% vs 90.8% (36.8 percentage points difference)
- Antonym zero-shot +FV: 26% vs 57.5% (31.5 percentage points difference)

The replication documentation ACKNOWLEDGES these discrepancies and provides explanations:
1. Smaller sample size (50 vs full test set)
2. No filtering to samples model gets correct on clean ICL
3. Potential seed differences

**Verdict: FAIL** - Results deviate significantly beyond 5% tolerance, even though directional trends match.

---

### DE2. Conclusion Consistency

**Original Conclusions (from plan.md):**
1. A small number of attention heads transport a compact representation (function vector) that is robust to changes in context
2. FVs can trigger task execution in zero-shot and natural text settings
3. FVs work best when added at early-middle layers (~L/3)
4. Function vectors contain information encoding the output space

**Replicated Conclusions (from documentation_replication.md):**
1. "Function vectors successfully transfer task knowledge" - matches original
2. "FVs recover performance in corrupted contexts" - matches original
3. "FVs don't harm clean ICL" - matches original
4. "Universal heads are effective" - matches original
5. "FVs are a real phenomenon that enables zero-shot task transfer" - matches original

**Assessment for DE2:**
The replicated conclusions are CONSISTENT with the original. The replication:
- Confirms the core claims about function vectors
- Acknowledges lower absolute numbers while validating the mechanism
- Does not contradict or omit essential claims from the original

**Verdict: PASS** - Conclusions are consistent in direction and interpretation.

---

### DE3. No External or Hallucinated Information

**Items to check:**
1. Are there invented findings not in the original?
2. Are there external references not supported by original documentation?
3. Are there hallucinated details?

**Assessment:**
The replicated documentation:
- Only reports results from experiments that were actually run
- Correctly references the paper citation (Todd et al., ICLR 2024)
- Correctly describes the methodology from the original repo
- Does not introduce unsupported claims
- The limitations section acknowledges what was NOT replicated

**Verdict: PASS** - No external or hallucinated information detected.
"""
print(analysis)


## Documentation Evaluation Analysis

### DE1. Result Fidelity

**Comparison of Results:**

**Antonym Task:**
| Metric | Original (plan.md) | Replicated | Deviation |
|--------|-------------------|------------|-----------|
| Shuffled-label baseline | 39.1% | 30.0% | -23.3% relative |
| Shuffled-label + FV | 90.8% | 54.0% | -40.5% relative |
| Zero-shot baseline | 5.5% | 0.0% | -100% relative |
| Zero-shot + FV | 57.5% | 26.0% | -54.8% relative |

**Country-Capital Task:**
| Metric | Original | Replicated | Note |
|--------|----------|------------|------|
| Zero-shot baseline | Not specified | 7.1% | - |
| Zero-shot + FV | 57-72% (natural text) | 83.3% | Exceeds original |

**Assessment for DE1:**
The replicated results show the same DIRECTIONAL trends as the original:
- Function vectors improve performance over baselines in all tested contexts
- The qualitative demonstrations (antonym: "static" → "dynamic", country-capital: correct capital) match expected behavior

HOWEVER, the absolu

In [16]:
# Create the evaluation output directory
eval_output_dir = '/net/scratch2/smallyan/function_vectors_eval/evaluation/new_replication_eval'
os.makedirs(eval_output_dir, exist_ok=True)
print(f"Created directory: {eval_output_dir}")

Created directory: /net/scratch2/smallyan/function_vectors_eval/evaluation/new_replication_eval


In [17]:
# Create documentation_evaluation_summary.md
documentation_eval_summary_md = """# Documentation Evaluation Summary

## Replicator-Documentation Evaluation

**Original Repository:** `/net/scratch2/smallyan/function_vectors_eval`  
**Replicated Documentation:** `/net/scratch2/smallyan/function_vectors_eval/evaluation/replications/documentation_replication.md`  
**Evaluation Date:** 2026-01-16

---

## Results Comparison

The replicated documentation reports results for the Function Vectors experiment on two tasks (Antonym and Country-Capital) evaluated on GPT-J 6B.

### Antonym Task Results

| Context | Original (plan.md) | Replicated | Deviation |
|---------|-------------------|------------|-----------|
| Shuffled-label baseline | 39.1% | 30.0% | -9.1pp |
| Shuffled-label + FV | 90.8% | 54.0% | -36.8pp |
| Zero-shot baseline | 5.5% | 0.0% | -5.5pp |
| Zero-shot + FV | 57.5% | 26.0% | -31.5pp |

### Country-Capital Task Results

| Context | Original | Replicated | Note |
|---------|----------|------------|------|
| Zero-shot + FV | 57-72% (natural text) | 83.3% | Exceeds original range |

The replicated results show **consistent directional effects** (function vectors improve performance across all contexts) but exhibit **substantial numerical deviations** from the original paper's reported metrics. The replication documentation acknowledges these discrepancies and attributes them to:
1. Smaller evaluation sample size (50 samples vs. full test set)
2. No filtering to samples the model gets correct on clean ICL
3. Potential differences in random seed

---

## Conclusions Comparison

The replicated documentation presents conclusions that are **consistent** with the original findings:

| Original Claim | Replicated Conclusion | Status |
|---------------|----------------------|--------|
| FVs transport compact task representations | "Function vectors successfully transfer task knowledge" | ✓ Consistent |
| FVs are robust to context changes | "FVs recover performance in corrupted contexts" | ✓ Consistent |
| FVs work in zero-shot and natural text | "FVs enable zero-shot task transfer" | ✓ Consistent |
| Universal heads are effective | "Pre-computed universal heads still produce effective function vectors" | ✓ Consistent |

Both documents conclude that function vectors are a real phenomenon that enables task knowledge transfer across different prompting contexts. The replicated conclusions do not contradict or meaningfully differ from the original.

---

## External or Hallucinated Information

**No external or hallucinated information was detected.** The replicated documentation:

- Only reports results from experiments that were actually executed
- Correctly references the original paper (Todd et al., ICLR 2024)
- Accurately describes the methodology from the original repository
- Explicitly acknowledges limitations and what was NOT replicated
- Does not introduce unsupported claims or invented findings

---

## Evaluation Checklist

| Criterion | Result | Rationale |
|-----------|--------|-----------|
| **DE1. Result Fidelity** | **FAIL** | Results deviate significantly beyond 5% tolerance. Antonym shuffled-label +FV shows 54% vs 90.8% original (-36.8pp), and zero-shot +FV shows 26% vs 57.5% (-31.5pp). |
| **DE2. Conclusion Consistency** | **PASS** | All conclusions are consistent with the original. The replication confirms the core mechanism and directional effects without contradicting essential claims. |
| **DE3. No External Information** | **PASS** | No external references, invented findings, or hallucinated details were introduced. The documentation accurately represents the executed experiments. |

---

## Final Verdict

**REVISION REQUIRED**

The replicated documentation demonstrates the correct methodology and confirms the qualitative behavior of function vectors, but the quantitative results deviate substantially from the original metrics beyond the acceptable 5% tolerance threshold. The directional effects are preserved, but the absolute accuracy numbers differ significantly.

### Recommendations for Revision:
1. Increase evaluation sample size to match the full test set
2. Apply the same evaluation filtering as the original paper (if applicable)
3. Verify random seed and experimental configuration alignment with original setup
4. Re-run experiments with matched configurations to achieve closer numeric alignment
"""

# Write to file
with open(os.path.join(eval_output_dir, 'documentation_evaluation_summary.md'), 'w') as f:
    f.write(documentation_eval_summary_md)

print("Written: documentation_evaluation_summary.md")
print(f"Path: {eval_output_dir}/documentation_evaluation_summary.md")

Written: documentation_evaluation_summary.md
Path: /net/scratch2/smallyan/function_vectors_eval/evaluation/new_replication_eval/documentation_evaluation_summary.md


In [18]:
# Create documentation_eval_summary.json
documentation_eval_summary_json = {
    "Checklist": {
        "DE1_ResultFidelity": "FAIL",
        "DE2_ConclusionConsistency": "PASS",
        "DE3_NoExternalInformation": "PASS"
    },
    "Rationale": {
        "DE1_ResultFidelity": "Results deviate significantly beyond 5% tolerance. Antonym task shuffled-label +FV shows 54% vs 90.8% original (-36.8 percentage points), zero-shot +FV shows 26% vs 57.5% (-31.5 percentage points). While directional effects are preserved, absolute numeric values fail the fidelity threshold.",
        "DE2_ConclusionConsistency": "All conclusions are consistent with the original. The replication confirms function vectors successfully transfer task knowledge, recover performance in corrupted contexts, and enable zero-shot task transfer - matching the core claims without contradiction or omission.",
        "DE3_NoExternalInformation": "No external references, invented findings, or hallucinated details were introduced. The documentation accurately reports only experiments that were executed and correctly references the original paper and methodology."
    }
}

# Write to file
with open(os.path.join(eval_output_dir, 'documentation_eval_summary.json'), 'w') as f:
    json.dump(documentation_eval_summary_json, f, indent=2)

print("Written: documentation_eval_summary.json")
print(f"Path: {eval_output_dir}/documentation_eval_summary.json")
print("\n" + "="*50)
print("JSON Contents:")
print(json.dumps(documentation_eval_summary_json, indent=2))

Written: documentation_eval_summary.json
Path: /net/scratch2/smallyan/function_vectors_eval/evaluation/new_replication_eval/documentation_eval_summary.json

JSON Contents:
{
  "Checklist": {
    "DE1_ResultFidelity": "FAIL",
    "DE2_ConclusionConsistency": "PASS",
    "DE3_NoExternalInformation": "PASS"
  },
  "Rationale": {
    "DE1_ResultFidelity": "Results deviate significantly beyond 5% tolerance. Antonym task shuffled-label +FV shows 54% vs 90.8% original (-36.8 percentage points), zero-shot +FV shows 26% vs 57.5% (-31.5 percentage points). While directional effects are preserved, absolute numeric values fail the fidelity threshold.",
    "DE2_ConclusionConsistency": "All conclusions are consistent with the original. The replication confirms function vectors successfully transfer task knowledge, recover performance in corrupted contexts, and enable zero-shot task transfer - matching the core claims without contradiction or omission.",
    "DE3_NoExternalInformation": "No external

In [19]:
# Verify that both files were written correctly
print("Verification of output files:")
print("="*50)

# List files in the output directory
files = os.listdir(eval_output_dir)
print(f"\nFiles in {eval_output_dir}:")
for f in files:
    filepath = os.path.join(eval_output_dir, f)
    size = os.path.getsize(filepath)
    print(f"  {f} ({size} bytes)")

# Read back and verify
print("\n" + "="*50)
print("Verification - documentation_evaluation_summary.md exists and is non-empty:")
with open(os.path.join(eval_output_dir, 'documentation_evaluation_summary.md'), 'r') as f:
    content = f.read()
    print(f"  Length: {len(content)} characters")
    print(f"  First 200 chars: {content[:200]}...")

print("\n" + "="*50)
print("Verification - documentation_eval_summary.json is valid JSON:")
with open(os.path.join(eval_output_dir, 'documentation_eval_summary.json'), 'r') as f:
    loaded_json = json.load(f)
    print(f"  Keys: {list(loaded_json.keys())}")
    print(f"  Checklist results: {loaded_json['Checklist']}")

Verification of output files:

Files in /net/scratch2/smallyan/function_vectors_eval/evaluation/new_replication_eval:
  documentation_eval_summary.json (1041 bytes)
  documentation_evaluation_summary.md (4327 bytes)

Verification - documentation_evaluation_summary.md exists and is non-empty:
  Length: 4319 characters
  First 200 chars: # Documentation Evaluation Summary

## Replicator-Documentation Evaluation

**Original Repository:** `/net/scratch2/smallyan/function_vectors_eval`  
**Replicated Documentation:** `/net/scratch2/small...

Verification - documentation_eval_summary.json is valid JSON:
  Keys: ['Checklist', 'Rationale']
  Checklist results: {'DE1_ResultFidelity': 'FAIL', 'DE2_ConclusionConsistency': 'PASS', 'DE3_NoExternalInformation': 'PASS'}


In [20]:
# Final summary output
print("="*70)
print("DOCUMENTATION EVALUATION COMPLETE")
print("="*70)

print("""
EVALUATION SUMMARY
==================

Original Repository: /net/scratch2/smallyan/function_vectors_eval
Replicated Documentation: /net/scratch2/smallyan/function_vectors_eval/evaluation/replications/documentation_replication.md

CHECKLIST RESULTS:
------------------
DE1. Result Fidelity:        FAIL
DE2. Conclusion Consistency: PASS
DE3. No External Information: PASS

FINAL VERDICT: REVISION REQUIRED
--------------------------------

The replicated documentation shows consistent directional effects (function 
vectors improve performance across all contexts) but exhibits substantial 
numerical deviations from the original paper's reported metrics, exceeding 
the 5% tolerance threshold.

Key Deviations:
- Antonym shuffled-label +FV: 54% vs 90.8% (36.8pp difference)
- Antonym zero-shot +FV: 26% vs 57.5% (31.5pp difference)

Conclusions remain consistent and no hallucinated information was introduced.

OUTPUT FILES:
-------------
1. /net/scratch2/smallyan/function_vectors_eval/evaluation/new_replication_eval/documentation_evaluation_summary.md
2. /net/scratch2/smallyan/function_vectors_eval/evaluation/new_replication_eval/documentation_eval_summary.json
""")

print("Task completed successfully.")

DOCUMENTATION EVALUATION COMPLETE

EVALUATION SUMMARY

Original Repository: /net/scratch2/smallyan/function_vectors_eval
Replicated Documentation: /net/scratch2/smallyan/function_vectors_eval/evaluation/replications/documentation_replication.md

CHECKLIST RESULTS:
------------------
DE1. Result Fidelity:        FAIL
DE2. Conclusion Consistency: PASS
DE3. No External Information: PASS

FINAL VERDICT: REVISION REQUIRED
--------------------------------

The replicated documentation shows consistent directional effects (function 
vectors improve performance across all contexts) but exhibits substantial 
numerical deviations from the original paper's reported metrics, exceeding 
the 5% tolerance threshold.

Key Deviations:
- Antonym shuffled-label +FV: 54% vs 90.8% (36.8pp difference)
- Antonym zero-shot +FV: 26% vs 57.5% (31.5pp difference)

Conclusions remain consistent and no hallucinated information was introduced.

OUTPUT FILES:
-------------
1. /net/scratch2/smallyan/function_vectors_